# Part 2.1 The rebooking agent in Studio

After this notebook you can run the finished Kea Air rebooking agent in LangSmith Studio
and point to each thing it does: call tools, plan, delegate, pause for approval, and
remember a passenger.

A passenger writes: *"My flight got cancelled."* The agent looks up the booking, explains
why, finds another flight, applies the compensation policy, asks a human before it issues
travel credit, and remembers the passenger's seat preference next time.

### Deep Agents or LangGraph?

[Part 1](../part1-langgraph/03_claims_workflow_in_studio.ipynb) was a **workflow**:
our code decided what ran next. This part is the **agent** column: the model runs a loop
and decides what to do next, calling tools until it is done. A rebooking conversation
can't be drawn in advance. The cause, the open seats, and what the passenger asks for all
change the path.

<img src="../../assets/workflows-vs-agent-light.svg" width="720">

<img src="../../assets/slides/runtime-framework-harness.jpg" width="720">

Deep Agents is the **harness** at the right of the slide. It sits on LangChain and
LangGraph and comes with a filesystem, subagents, and middleware (and an opt-in planning tool), so you
configure an agent instead of wiring a graph.

| Reach for | When |
|---|---|
| **Deep Agents** | Open-ended, long-running, planning-heavy work that needs files, memory, and subagents |
| **LangGraph** | You want to draw the graph yourself: fixed steps, branches you control, parallel fan-out |

They compose. Deep Agents is built on LangGraph, so checkpointers, the Store, interrupts,
and Studio work the same way. Details:
[LangChain, LangGraph, and Deep Agents](https://docs.langchain.com/oss/concepts/products).

In [ ]:
from kea.config import setup

setup("kea-rebooking-local")

`setup` loads your LangSmith key and, if you had to type it in, saves it to `.env` so the
server you start below can read it.

## See it

This notebook has no code to run. You drive the agent from Studio.

**1. Start the server.** In a terminal at the repo root:

```bash
mise run dev
```

**2. Open Studio.** Go to <https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024>. In Chrome, allow "Local network access" when it asks.
Pick the **`rebooking`** graph in the top-left menu.

<img src="../../assets/screenshots/p2-01-studio-graph.png" alt="Studio with the rebooking graph selected: the graph view in the middle and the empty chat on the right." width="900">

**3. Send the cancelled-flight message** in a new thread:

```text
My flight KA 1182 was cancelled. Confirmation ABC123, last name Example. Rebook me and
give me what I'm owed.
```

`ABC123` and `Example` are Alex Example's booking from `data/rebooking/bookings.json`.
KA 1182 was cancelled for a mechanical problem.

**4. Watch the tool calls.** Each one appears as a step as it runs. You should see
`get_booking`, then `search_flights`, then `rebook`. Click a step to read its inputs and
result. `web_search_exa` may appear too, when the agent checks airport conditions.

<img src="../../assets/screenshots/p2-01-tool-calls.png" alt="Studio's thread view after the first message: the getbooking, searchflights, and rebook tool calls expanded, with rebook's receipt visible." width="900">

**5. Find the subagent.** Look for a `task` call to **`policy-checker`**. Expand it: the
subagent read the rebooking policy in its own context and sent back a short answer. Alex's
delay is 145 minutes for a mechanical cancellation, so the answer is a $100 credit.

<img src="../../assets/screenshots/p2-01-subagent.png" alt="The task call to policy-checker expanded, with the subagent's readfile of /skills/rebooking-policy/SKILL.md and its short answer." width="900">

**6. Approve the credit.** The run stops at `issue_credit` and shows an approval card with
the proposed arguments: `ABC123`, `100`, and a reason. Nothing has been issued yet. Click
**Approve**. The agent finishes with a reply to the passenger.

<img src="../../assets/screenshots/p2-01-credit-approval.png" alt="The interrupt card for issuecredit with the proposed arguments and the Approve, Edit, and Reject buttons." width="900">

To see the other two decisions, send the message again in a fresh thread and try **Edit**
(change the amount) or **Reject** (add a reason the agent will read).

**7. Look at the state and files.** In the same thread, send:

```text
Please save a one-paragraph case note for this booking to /notes/abc123.md.
```

Open the **State** panel. The note is in `files`. These files live in the thread, so they
are gone in a new thread.

<img src="../../assets/screenshots/p2-01-case-note-files.png" alt="The writefile call that saves the case note /notes/abc123.md." width="900">

There is no todo list. Deep Agents' planning tool (`write_todos`) is opt-in since v0.7,
and this agent doesn't turn it on.

**8. Save a preference.** In the same thread, send:

```text
By the way, I always want an aisle seat. Please remember that.
```

The agent writes the preference to `/memories/preferences.md`. Unlike the case note, this
file goes to the Store and outlives the thread.

<img src="../../assets/screenshots/p2-01-memory-write.png" alt="The editfile call to /memories/preferences.md with its content, in the thread view." width="900">

**9. Start a second thread.** Click **New thread**. Nothing from the first thread is in its
messages. Send:

```text
Hi, it's Alex Example again. Confirmation ABC123, flight KA 1182 was cancelled. Please
rebook me.
```

The agent lists `/memories/`, reads `preferences.md`, and calls `rebook` with
`seat_preference='aisle'`, even though the booking says `window`. It will propose another
credit. Approve it.

<img src="../../assets/screenshots/p2-01-memory-read.png" alt="The second thread: ls /memories/, readfile /memories/preferences.md, then rebook(..., seatpreference='aisle')." width="900">

To start clean, stop the server, delete the `.langgraph_api/` folder, and run
`mise run dev` again. That folder is where the server keeps the Store.

## Build it

You build this agent one capability at a time. Each notebook starts from the finished
result, then writes the code for that one piece.

| Notebook | You build | You saw it in Studio as |
|---|---|---|
| [2.2 Tools](02_tools.ipynb) | Typed functions the model can call | `get_booking`, `search_flights`, `rebook` |
| [2.3 MCP](03_mcp.ipynb) | Web search from a remote MCP server | `web_search_exa` |
| [2.4 Backends](04_backends.ipynb) | Where the agent's files live | `files` in the State panel |
| [2.5 Memory](05_memory.ipynb) | What the agent remembers across threads | `/memories/preferences.md` |
| [2.6 Skills](06_skills.ipynb) | Policy loaded only when needed | `read_file` of the policy |
| [2.7 Subagents](07_subagents.ipynb) | Work done in a separate context | the `policy-checker` task |
| [2.8 Middleware](08_middleware.ipynb) | Approval, privacy, and limits | the `issue_credit` approval card |
| [2.9 Assertion evals](09_assertion_evals.ipynb) | A test set you can run on any model | experiments in LangSmith |

## Under the hood

- **The loop.** Call the model, run the tools it asked for, feed the results back, and
  repeat until the model answers without a tool call. Everything you saw is something
  added to that loop.
- **What Deep Agents adds.** Every agent gets file tools (`ls`, `read_file`, `write_file`,
  `edit_file`, `glob`, `grep`) and a `task` tool that hands work to a subagent. You didn't
  write those. A planning tool (`write_todos`, via `TodoListMiddleware`) is opt-in.
- **What Studio talks to.** `langgraph.json` lists two graphs, `claims` and `rebooking`.
  `mise run dev` starts an Agent Server on port 2024 that serves both. The server supplies
  the checkpointer (thread history) and the Store (memory across threads), which is why
  the second thread could read the first thread's preference.
- **The code.** The agent is `kea.rebooking.agent.agent`. Open that file after the
  workshop. It is about a hundred lines.

Docs: [LangSmith Studio](https://docs.langchain.com/langsmith/studio),
[Deep Agents overview](https://docs.langchain.com/oss/python/deepagents/overview).

## Recap

- Use **Deep Agents** for open-ended work with files, planning, and subagents. Use
  **LangGraph** when you want to own the graph.
- Studio shows every tool call, the state, the subagent, and the approval pause.
- Memory across threads comes from the Store, which the server supplies.

**Next:** [Part 2.2 Tools](02_tools.ipynb)